# Pipeline KYC — Inventaire des documents & extraction du justificatif d'identité

**Objectif de ce notebook**

1. Dézipper `kyc_documents.zip` et repérer, dans chaque dossier client, les 5 documents utiles parmi tous
   ceux présents : `JUSTIFICATIF IDENTITE.PDF`, `JUSTIFICATIF DOMICILE.PDF`, `CONVENTION COMPTE.PDF`,
   `FATCA.PDF`, `CARTON SIGNATURE.PDF`.
2. Produire un **rapport d'inventaire** (CSV) indiquant, pour chaque client, quels documents sont présents.
3. Déterminer la **liste des clients cibles** : ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` existe.
4. Pour ces clients, **extraire automatiquement l'information** du `JUSTIFICATIF IDENTITE.PDF` (document
   scanné, potentiellement multi-page, multilingue, mal orienté ou de mauvaise qualité) à l'aide du modèle
   **Qwen3.5** (vision-langage) hébergé localement sur votre ModelHub Domino — aucune donnée ne sort de
   votre environnement.

**Hypothèses et choix retenus** (tout est regroupé en Partie 1 pour être ajusté facilement) :

- La liste des fichiers cibles contient `CARTON SIGNATURE.PDF` : l'énoncé initial mentionnait
  *"CARTON SIGNATUTE.PDF"*, probablement une coquille — corrigez `TARGET_FILES` si le nom réel diffère
  dans vos dossiers.
- Le `config.json` fourni indique `"model_type": "qwen3_5"`, c'est-à-dire l'architecture **Qwen3.5**
  (multimodale texte/image/vidéo). Ce nom diffère légèrement de celui du dossier ModelHub ("Qwen3.8"),
  probablement une convention de nommage interne : le code se base sur l'architecture réellement déclarée
  dans le config.json, pas sur le nom du dossier.
- La comparaison des noms de fichiers est **insensible à la casse** (les scans bancaires ont rarement une
  casse homogène), et chaque dossier client est parcouru **récursivement**.
- Seul `JUSTIFICATIF IDENTITE.PDF` est traité par le modèle dans ce notebook, comme demandé. Les 4 autres
  types de documents sont inventoriés ; le même patron de code (Parties 4 à 8) pourra leur être appliqué
  ensuite.
- Le traitement par lot est **reprenable** : chaque client traité produit un fichier JSON individuel, donc
  une interruption (crash, timeout) ne fait pas perdre le travail déjà effectué.
- Pour la conversion PDF → image, on utilise **PyMuPDF** plutôt que `pdf2image`/Poppler : aucune dépendance
  système (binaire externe) à installer, ce qui est plus robuste dans un environnement managé comme Domino.
- Ce notebook ne fige pas de numéros de version exacts pour `torch`/`transformers` (écosystème qui évolue
  vite) : il installe une **version plancher connue pour fonctionner avec Qwen3.5**, puis **enregistre les
  versions réellement installées** dans un fichier (`environnement_installe.txt`) pour la traçabilité.
- Le modèle est chargé via la classe générique **`AutoModelForImageTextToText`** (et non la classe
  spécifique `Qwen3_5ForConditionalGeneration`) : c'est l'approche que vous avez adoptée, et c'est aussi
  celle recommandée par la fiche officielle du modèle sur Hugging Face — elle résout automatiquement la
  bonne architecture à partir de `config.json`, sans dépendre d'un nom de classe figé dans ce notebook.
- Le prétraitement (Partie 4/5) est **systématique et dans un ordre précis** — pas une case à cocher au
  cas par cas — car en réalité, les scans bancaires sont mal orientés ET de mauvaise qualité, pas l'un ou
  l'autre : chaque page est d'abord ramenée à une résolution sûre pour le modèle, puis son contraste est
  amélioré, puis sa rotation franche corrigée, et enfin son inclinaison résiduelle corrigée en dernier.
- **Diagnostic révisé du `!!!!!!!!` (important)** : ce symptôme n'est pas une « mauvaise lecture » du
  document. Une suite d'un seul caractère répété est la signature de **logits `NaN`** : quand le calcul
  produit des valeurs invalides, l'`argmax` retombe toujours sur le même identifiant de token (le token 0,
  soit `!` dans le vocabulaire Qwen), et ce caractère est répété jusqu'à `max_new_tokens`. C'est cohérent
  avec le crash CUDA que vous aviez eu **dans le noyau FP8** (`w8a8_block_fp8_matmul`), et cela explique
  pourquoi ni `MAX_IMAGE_DIMENSION_MODEL=768` ni `MAX_NEW_TOKENS_EXTRACTION=10000` n'ont pu aider : la
  cause n'est pas dans l'image. Le notebook ajoute donc (a) un paramètre **`MODEL_LOAD_MODE`** (Partie 1,
  `"bf16"` par défaut) qui contourne entièrement le calcul FP8, et (b) un **escalier de diagnostic**
  (Partie 5bis-A) qui teste le modèle sans image, puis sur une image parfaite, puis sur votre scan — le
  premier niveau qui échoue désigne la cause, au lieu de la deviner.
- Un plafond de résolution (`MAX_IMAGE_DIMENSION_MODEL`, Partie 1) reste appliqué avant tout appel au
  modèle. Ce n'était pas la cause chez vous, mais cela accélère l'inférence (moins de « tokens image ») et
  prévient un autre mode d'échec réellement rapporté sur les entrées à très haute résolution.
- Un diagnostic de répartition GPU/CPU du modèle est affiché juste après son chargement (Partie 5) : un
  déchargement partiel sur CPU, même minime, est généralement le facteur le plus déterminant pour la
  lenteur d'un traitement par lot (Partie 7) — bien avant les autres optimisations apportées.
- `TRUST_REMOTE_CODE` (Partie 1, par défaut `True`) autorise le téléchargement du noyau de calcul FP8
  optimisé (RedHatAI, via la librairie `kernels`) plutôt que de basculer silencieusement sur une
  implémentation de repli qui a fait planter CUDA lors de votre test (`AcceleratorError` dans
  `w8a8_block_fp8_matmul`) — voir l'explication complète en Partie 5 et l'alternative bf16 si votre
  politique de sécurité impose de le désactiver.
- **Sur votre dernier test** (`MAX_IMAGE_DIMENSION_MODEL=768`, bien en dessous du seuil qui posait
  problème dans les rapports communautaires, et toujours une réponse dégénérée) : ceci confirme que la
  résolution d'image n'est très probablement PAS (ou plus, si `TRUST_REMOTE_CODE` a résolu le crash CUDA)
  la cause principale chez vous. Ce notebook ajoute donc une chaîne de techniques **indépendantes du
  modèle** (Tesseract OCR : orientation + extraction de texte brut) qui, en plus d'améliorer la robustesse
  sur des scans réels (couleur/N&B, plusieurs pages, mal orientés, mauvaise qualité), servent de
  **diagnostic automatique** : si Tesseract lit un texte cohérent sur une image et que le modèle continue
  de répondre par des caractères répétés, le résultat le dit explicitement — cela pointe vers le modèle/
  l'environnement, pas vers le document (voir Partie 4 pour le détail de chaque étape, et Partie 5bis pour
  voir ce diagnostic en action sur un client réel).

## Partie 0 — Installation des librairies

Installez dans cet ordre : d'abord les librairies de fichiers/images (légères, sans risque), puis la pile
modèle (`transformers`, `accelerate`, `compressed-tensors`), et enfin `torch` — à adapter impérativement à
la version CUDA de votre environnement Domino (voir commentaire ci-dessous). Si `torch` est déjà préinstallé
dans votre image Domino (fréquent sur les environnements GPU), vous pouvez sauter cette ligne.

In [ ]:
# --- Traitement de fichiers / PDF / images (aucune dépendance système requise) ---
%pip install -q "pymupdf>=1.26.0"                   # rendu des pages PDF en images ; respecte la rotation déclarée dans le PDF
%pip install -q "pillow>=10.4.0"                    # manipulation d'images
%pip install -q "opencv-python-headless>=4.10.0"    # redressement (deskew) + contraste ; "headless" = pas de dépendance GUI/libGL
%pip install -q "numpy>=1.26.0"
%pip install -q "pandas>=2.2.0"                     # rapport d'inventaire, consolidation des résultats
%pip install -q "tqdm>=4.66.0"                      # barre de progression du traitement par lot

# --- Pile modèle : Qwen3.5 (vision-langage), quantifié FP8 ---
%pip install -q "transformers>=5.8.0"               # le model_type "qwen3_5" est supporté à partir de la 5.8 ; privilégiez la dernière version stable
%pip install -q "accelerate>=0.34.0"                # requis pour device_map="auto" (répartition automatique sur GPU)
%pip install -q "compressed-tensors>=0.7.0"         # requis pour décoder les poids quantifiés FP8 du checkpoint (cf. quantization_config, Partie 5)

# --- OCR complémentaire (Tesseract) : orientation + texte brut, INDÉPENDANT du modèle Qwen3.5 ---
# Utile à deux titres : (1) robustesse réelle sur des scans mal orientés/de mauvaise qualité (voir Partie 4),
# (2) DIAGNOSTIC -- si Tesseract lit un texte cohérent sur une image alors que le modèle répond par des
# caractères répétés, cela indique un problème côté modèle/environnement, pas côté document.
# Le binaire système `tesseract-ocr` nécessite les droits d'installation apt (root) : si votre environnement
# Domino ne les autorise pas, demandez à votre équipe infra de le pré-installer dans l'image, OU laissez
# simplement `ENABLE_OCR_ASSIST = False` (Partie 1) -- le reste du notebook fonctionne sans, en se rabattant
# sur les techniques précédentes (détection d'orientation par le modèle uniquement).
!apt-get install -y -qq tesseract-ocr tesseract-ocr-fra tesseract-ocr-ara 2>/dev/null || echo "apt-get indisponible/non autorisé ici -- voir le commentaire ci-dessus"
%pip install -q "pytesseract>=0.3.10"

# --- PyTorch : à adapter à VOTRE version CUDA (vérifiez avec `!nvidia-smi`) ---
# Décommentez et ajustez l'URL d'index si torch n'est pas déjà présent dans votre environnement Domino, par ex. :
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu124

# --- Optionnel : accélère l'attention hybride (Gated DeltaNet) de Qwen3.5 ---
# Sans ces paquets, le modèle fonctionne normalement mais bascule automatiquement sur un mode de repli
# PyTorch plus lent pour SES COUCHES D'ATTENTION LINÉAIRE spécifiquement. Leur compilation nécessite un
# toolchain CUDA correspondant exactement à votre torch : à tenter seulement si la vitesse d'inférence
# pose problème, sinon inutile de les installer.
# (Les couches d'attention "classique" du modèle utilisent déjà le backend natif et rapide de PyTorch —
# attn_implementation="sdpa", activé automatiquement en Partie 5 — qui ne nécessite aucun paquet en plus.)
# %pip install -q -U kernels
# %pip install -q causal-conv1d --no-build-isolation

## Imports groupés

Toutes les librairies utilisées dans ce notebook, importées une seule fois ici.

In [ ]:
# Bibliothèque standard
import os
import re
import io
import sys
import json
import time
import shutil
import zipfile
import logging
import platform
import subprocess
from pathlib import Path
from datetime import datetime

# Traitement de données / fichiers / images
import numpy as np
import pandas as pd
import cv2
import pymupdf
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

# Modèle
import torch
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText

# OCR complémentaire (optionnel — voir ENABLE_OCR_ASSIST, Partie 1)
try:
    import pytesseract
    pytesseract.get_tesseract_version()  # lève une exception si le binaire système est absent, pas juste le paquet Python
    TESSERACT_AVAILABLE = True
except Exception as _e:
    TESSERACT_AVAILABLE = False
    _TESSERACT_IMPORT_ERROR = _e

print("Toutes les librairies ont été importées avec succès.")
print(f"Tesseract OCR disponible : {TESSERACT_AVAILABLE}" + ("" if TESSERACT_AVAILABLE else f" ({_TESSERACT_IMPORT_ERROR})"))

## Partie 1 — Configuration

Tous les paramètres modifiables du pipeline sont centralisés ici : chemins, liste des fichiers cibles,
options de prétraitement d'image. C'est le seul endroit à modifier pour adapter le notebook à votre
environnement exact.

In [ ]:
# ============================== CHEMINS ==============================
ZIP_PATH = Path("kyc_documents.zip")                       # <-- à adapter : emplacement réel du zip dans Domino
WORK_DIR = Path("kyc_pipeline_workdir")                     # tous les fichiers produits par ce notebook y seront rangés

RAW_EXTRACT_DIR  = WORK_DIR / "01_extraction_brute"          # dézippage complet et brut
FILTERED_DIR     = WORK_DIR / "02_documents_cibles"           # uniquement les 5 fichiers utiles, par client
RESULTS_DIR      = WORK_DIR / "03_resultats_identite"         # un fichier JSON par client traité (reprenable)

REPORT_PATH            = WORK_DIR / "rapport_inventaire_kyc.csv"
TARGET_CLIENTS_PATH    = WORK_DIR / "liste_clients_cibles.csv"
COMBINED_RESULTS_JSON  = WORK_DIR / "resultats_extraction_identite.json"
COMBINED_RESULTS_CSV   = WORK_DIR / "resultats_extraction_identite.csv"
LOG_PATH               = WORK_DIR / "pipeline_kyc.log"
ENV_SNAPSHOT_PATH      = WORK_DIR / "environnement_installe.txt"

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"  # chemin fourni

# ============================== FICHIERS CIBLES ==============================
# Comparaison insensible à la casse (voir normalize() en Partie 2) : la casse ci-dessous n'a pas besoin de
# correspondre exactement à celle des fichiers réels sur le disque.
TARGET_FILES = [
    "JUSTIFICATIF IDENTITE.PDF",
    "JUSTIFICATIF DOMICILE.PDF",
    "CONVENTION COMPTE.PDF",
    "FATCA.PDF",
    "CARTON SIGNATURE.PDF",     # "SIGNATURE" ; remplacez par "SIGNATUTE" si c'est réellement ce nom-là chez vous
]
IDENTITY_FILE = "JUSTIFICATIF IDENTITE.PDF"    # doit être un élément exact de TARGET_FILES ci-dessus

# ============================== PARAMÈTRES DE TRAITEMENT ==============================
PDF_RENDER_DPI               = 300     # résolution de rendu PDF -> image. resize_for_model (Partie 4)
                                        # réduit ensuite cette image avant l'envoi au modèle (voir
                                        # MAX_IMAGE_DIMENSION_MODEL) : cette valeur n'affecte donc que la
                                        # qualité de l'image intermédiaire et le temps de rendu PDF, pas la
                                        # résolution réellement vue par le modèle.
MAX_IMAGE_DIMENSION_MODEL    = 1568    # plafond de résolution (plus grand côté, en pixels) AVANT tout appel
                                        # au modèle. Ce n'est PAS une option de confort : des images plus
                                        # grandes ont été signalées par la communauté Hugging Face comme
                                        # provoquant une réponse dégénérée de Qwen3.5 sur les entrées image
                                        # (voir resize_for_model, Partie 4). Diminuez cette valeur (ex. 1024
                                        # ou 768) si le problème persiste malgré tout — voir la checklist de
                                        # dépannage en Partie 5bis.
ENABLE_ORIENTATION_CHECK     = True    # corrige les rotations franches (90/180/270°) : Tesseract d'abord, modèle en repli
ENABLE_SKEW_CORRECTION       = True    # corrige les légères inclinaisons (quelques degrés) via OpenCV
ENABLE_CONTRAST_ENHANCEMENT  = True    # améliore le contraste des scans de mauvaise qualité (CLAHE)
ENABLE_DENOISING             = False   # débruitage supplémentaire pour scans TRÈS dégradés -- plus lent (CPU),
                                        # à activer seulement si ENABLE_CONTRAST_ENHANCEMENT seul ne suffit pas
MAX_NEW_TOKENS_EXTRACTION    = 2048    # longueur max. de la réponse du modèle pour l'extraction structurée
FORCE_REPROCESS              = False   # True = retraite même les clients déjà traités (sinon reprise automatique)

# ============================== LECTURE ROBUSTE DES PDF (multi-pages, qualité variable) ==============================
TEXT_LAYER_MIN_CHARS = 80              # nb minimal de caractères PAR PAGE pour considérer qu'un PDF possède une
                                        # vraie couche texte exploitable (voir extract_pdf_text_layer, Partie 4).
                                        # En dessous, on considère qu'il s'agit d'un vrai scan à traiter en image.
PREFER_TEXT_LAYER    = True            # True = si une couche texte native existe, l'utiliser directement plutôt
                                        # que de passer par le modèle (beaucoup plus rapide, exact, et insensible
                                        # au problème de réponse dégénérée). Mettez False pour forcer systéma-
                                        # tiquement le passage par la vision du modèle.
MAX_PAGES_PER_CALL   = 4               # nb max de pages envoyées au modèle en UN SEUL appel. Au-delà, le document
                                        # est découpé en plusieurs appels puis les résultats sont fusionnés
                                        # (voir merge_extractions, Partie 6) : un grand nombre d'images dans un
                                        # même appel augmente fortement la mémoire GPU et la latence, et dégrade
                                        # la qualité des réponses sur beaucoup de modèles.
RETRY_RESOLUTIONS    = [768, 512]      # en cas de réponse dégénérée, nouvelles tentatives à ces résolutions
                                        # (plus basses) avant d'abandonner. Liste vide = aucune nouvelle tentative.

# ============================== OCR COMPLÉMENTAIRE (Tesseract, indépendant du modèle) ==============================
ENABLE_OCR_ASSIST = True               # False si tesseract-ocr n'a pas pu être installé (Partie 0) ou si vous
                                        # préférez vous en passer -- le reste du pipeline fonctionne sans (repli
                                        # automatique sur le modèle seul pour l'orientation, pas d'indice OCR
                                        # dans le prompt, pas de diagnostic croisé en cas de réponse dégénérée)
OCR_LANGUAGES = "fra+ara"              # langues attendues sur les documents (contexte algérien bilingue) ;
                                        # ajoutez "+eng" si des documents anglais sont possibles
OCR_ORIENTATION_CONFIDENCE_THRESHOLD = 2.0   # confiance minimale (échelle Tesseract OSD) pour faire confiance à
                                        # sa détection d'orientation plutôt que de se rabattre sur le modèle ;
                                        # calibré empiriquement (voir Partie 4) -- à ajuster sur vos vrais
                                        # documents si les résultats semblent peu fiables

# ============================== SÉCURITÉ / CODE DISTANT (noyaux de calcul FP8) ==============================
# Le chargement d'un checkpoint quantifié FP8 (compressed-tensors) peut tenter de télécharger et D'EXÉCUTER un
# noyau de calcul optimisé (kernel CUTLASS) depuis un dépôt Hugging Face tiers (ex. "RedHatAI/quantization"),
# via la librairie `kernels`. Depuis son passage aux "éditeurs de confiance", `kernels` n'autorise PAR DÉFAUT
# que les dépôts explicitement approuvés par Hugging Face ; sinon, le chargement du kernel échoue avec un
# avertissement ("could not verify publisher trust status") et bascule silencieusement vers une implémentation
# de repli (Triton) plus lente ET, dans certains environnements, instable au point de faire planter CUDA en
# cours de génération (voir la checklist de dépannage, Partie 5bis, si vous rencontrez une erreur
# "AcceleratorError"/"cudaErrorUnknown").
# TRUST_REMOTE_CODE=True autorise ce téléchargement/exécution de code tiers — une VRAIE décision de sécurité,
# pas un simple réglage de confort : à faire valider par votre équipe sécurité/infra avant un déploiement en
# production dans un contexte bancaire, même si l'éditeur concerné ici (RedHatAI, l'organisation Red Hat) est
# une source réputée légitime. Passez à False si votre politique ne permet pas cette exécution de code distant
# — voir l'alternative sans noyau optimisé (chargement en bf16) proposée en Partie 5 dans ce cas.
TRUST_REMOTE_CODE = True

# ============================== PRÉCISION DE CHARGEMENT DU MODÈLE ==============================
# ⚠️ PARAMÈTRE LE PLUS IMPORTANT DE CE NOTEBOOK POUR VOTRE PROBLÈME ACTUEL.
# Une réponse constituée d'un seul caractère répété ("!!!!!!!!") n'est PAS une "mauvaise réponse" du modèle :
# c'est la signature classique de valeurs NaN/Inf dans le calcul. Les logits deviennent NaN, l'argmax retombe
# alors systématiquement sur le même identifiant de token (le token 0, qui correspond à "!" dans le
# vocabulaire Qwen), et le modèle "écrit" donc ce caractère jusqu'à épuisement de max_new_tokens. Ce symptôme
# est bien documenté sur les chemins de calcul quantifiés instables (cf. vllm-project/vllm issue #24025 avec
# un Qwen3 quantifié, et les nombreux rapports de NaN en float16 sur les modèles Qwen-VL).
#   -> Conséquence directe : ce n'est ni un problème de qualité de scan, ni d'orientation, ni de résolution
#      d'image. C'est pourquoi passer MAX_IMAGE_DIMENSION_MODEL à 768 n'a rien changé chez vous, et c'est
#      cohérent avec le crash CUDA observé précédemment DANS le noyau FP8 (w8a8_block_fp8_matmul).
#   -> Augmenter MAX_NEW_TOKENS_EXTRACTION (vous étiez passé à 10000) ne peut pas aider non plus : cela rend
#      seulement la suite de "!" plus longue et le traitement plus lent. 2048 suffit largement pour le JSON
#      attendu ici.
#
# Valeurs possibles :
#   "bf16"  (DÉFAUT RECOMMANDÉ ICI) : déquantifie le checkpoint vers bfloat16 au chargement et contourne
#           ENTIÈREMENT le chemin de calcul FP8 (DeepGEMM ET Triton). C'est le réglage qui a le plus de
#           chances de résoudre votre problème. Coût : environ 2x la mémoire GPU du FP8 -- surveillez le
#           diagnostic de répartition GPU/CPU affiché en Partie 5.
#   "fp8"   : conserve la quantification FP8 du checkpoint (dtype="auto"). Plus économe en mémoire, mais
#           c'est le chemin sur lequel le crash CUDA est survenu. À retenter seulement après avoir confirmé
#           que TRUST_REMOTE_CODE=True charge bien le noyau optimisé (aucun avertissement CUTLASS).
#   "fp16"  : À ÉVITER sur cette famille de modèles -- float16 est une cause connue et fréquente de NaN sur
#           les modèles Qwen-VL (entraînés en bfloat16). Proposé uniquement pour un GPU ancien incapable de
#           faire du bfloat16, et à ne pas privilégier si "!!!!" est justement votre symptôme.
MODEL_LOAD_MODE = "bf16"

# ============================== INITIALISATION ==============================
for d in (WORK_DIR, RAW_EXTRACT_DIR, FILTERED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
    force=True,  # évite les handlers dupliqués si la cellule est réexécutée
)
logger = logging.getLogger("kyc_pipeline")
logger.info("Configuration chargée. Répertoire de travail : %s", WORK_DIR.resolve())

# Remarque « protection des données » : les logs ne contiennent volontairement que des identifiants client
# et des statuts techniques — jamais les données personnelles extraites elles-mêmes.

Vérification de l'environnement (versions installées, GPU disponible) et sauvegarde d'un instantané des
versions réellement présentes, pour la traçabilité / l'audit (utile en contexte bancaire réglementé).

In [ ]:
print(f"Date d'exécution        : {datetime.now().isoformat(timespec='seconds')}")
print(f"Python                  : {sys.version.split()[0]} ({platform.system()} {platform.release()})")
print(f"PyTorch                 : {torch.__version__}")
print(f"Transformers            : {transformers.__version__}")
print(f"CUDA disponible         : {torch.cuda.is_available()}")
print(f"Tesseract OCR disponible : {TESSERACT_AVAILABLE}  (ENABLE_OCR_ASSIST = {ENABLE_OCR_ASSIST})")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i} : {props.name} — {props.total_memory / 1e9:.1f} Go")
else:
    print("Aucun GPU détecté. L'inférence sur un modèle de cette taille sera très lente, voire impraticable, sur CPU.")

with open(ENV_SNAPSHOT_PATH, "w", encoding="utf-8") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
logger.info("Snapshot de l'environnement sauvegardé -> %s", ENV_SNAPSHOT_PATH)

## Partie 2 — Dézippage et inventaire des documents KYC

On dézippe `kyc_documents.zip`, puis pour **chaque dossier client**, on recherche les 5 fichiers cibles
(recherche récursive, insensible à la casse). Les fichiers trouvés sont copiés dans une arborescence propre
(`FILTERED_DIR/<client_id>/<nom_canonique>.PDF`), et leur présence/absence est consignée dans un tableau
qui sera sauvegardé en CSV.

In [ ]:
def normalize(name: str) -> str:
    """Normalise un nom de fichier pour une comparaison insensible à la casse et aux espaces superflus."""
    return name.strip().upper()

TARGET_FILES_NORM = {normalize(f): f for f in TARGET_FILES}

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Archive introuvable : {ZIP_PATH.resolve()}. Vérifiez ZIP_PATH dans la cellule de configuration (Partie 1)."
    )

if RAW_EXTRACT_DIR.exists():
    shutil.rmtree(RAW_EXTRACT_DIR)
RAW_EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(RAW_EXTRACT_DIR)
logger.info("Archive dézippée -> %s", RAW_EXTRACT_DIR.resolve())

# Le zip peut soit contenir directement les dossiers clients, soit un unique dossier racine qui les englobe :
# on détecte automatiquement le bon niveau.
entries = list(RAW_EXTRACT_DIR.iterdir())
DATA_ROOT = entries[0] if (len(entries) == 1 and entries[0].is_dir()) else RAW_EXTRACT_DIR

client_folders = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir())
logger.info("%d dossier(s) client détecté(s) sous %s", len(client_folders), DATA_ROOT)

In [ ]:
rows = []
for client_dir in tqdm(client_folders, desc="Inventaire des documents"):
    client_id = client_dir.name
    all_files = [p for p in client_dir.rglob("*") if p.is_file()]   # recherche récursive

    found = {}  # nom_cible_canonique -> chemin réel trouvé
    for f in all_files:
        canonical = TARGET_FILES_NORM.get(normalize(f.name))
        if canonical:
            found[canonical] = f

    row = {"client_id": client_id}
    for target in TARGET_FILES:
        row[target] = target in found
    row["nb_documents_cibles_trouves"] = len(found)
    row["dossier_complet"] = len(found) == len(TARGET_FILES)
    rows.append(row)

    if found:
        dest_dir = FILTERED_DIR / client_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        for canonical_name, src_path in found.items():
            shutil.copy2(src_path, dest_dir / canonical_name)

inventory_df = pd.DataFrame(rows).set_index("client_id").sort_index()
inventory_df.to_csv(REPORT_PATH, encoding="utf-8-sig")

logger.info("Rapport d'inventaire sauvegardé -> %s", REPORT_PATH.resolve())
print(f"\n{len(inventory_df)} client(s) au total.")
for target in TARGET_FILES:
    print(f"  - {target:<32} présent chez {int(inventory_df[target].sum())} client(s)")
print(f"  - Dossiers complets (5/5)         : {int(inventory_df['dossier_complet'].sum())}")

inventory_df

## Partie 3 — Liste des clients cibles

Les **clients cibles** sont ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` a été trouvé : ce sont eux qui
seront traités dans la suite du notebook.

In [ ]:
target_clients = inventory_df.index[inventory_df[IDENTITY_FILE]].tolist()

pd.Series(target_clients, name="client_id").to_csv(TARGET_CLIENTS_PATH, index=False, encoding="utf-8-sig")
logger.info("%d client(s) cible(s) -> %s", len(target_clients), TARGET_CLIENTS_PATH.resolve())

print(f"{len(target_clients)} client(s) cible(s) sur {len(inventory_df)} :")
print(target_clients)

## Partie 4 — Lire n'importe quel scan : couleur/N&B, mauvaise qualité, plusieurs pages, mal orienté

Cette partie assemble une chaîne de techniques **complémentaires**, dont plusieurs sont volontairement
**indépendantes du modèle Qwen3.5** (elles reposent sur OpenCV et Tesseract OCR, qui tournent sur CPU).
Deux bénéfices concrets à cela : (1) le pipeline reste robuste même quand le modèle a un problème (comme
celui que vous avez rencontré), et (2) ça permet un **diagnostic automatique** capable de distinguer
« le document est illisible » de « le modèle ne répond pas correctement » : les deux donnent le même
symptôme en surface (réponse dégénérée) mais n'ont ni la même cause, ni le même remède.

**Les 7 étapes**, appliquées à chaque page dans cet ordre précis (assemblées dans `preprocess_page`,
Partie 5) :

| # | Étape | Fonction | Dépend du modèle ? |
|---|-------|----------|---------------------|
| 0 | **Couche texte native du PDF** (raccourci : évite tout le reste) | `extract_pdf_text_layer` | Non |
| 1 | Rendu PDF → image | `pdf_to_images` | Non |
| 2 | Plafond de résolution | `resize_for_model` | Non |
| 3 | Détection couleur / N&B | `detect_color_mode` | Non |
| 4 | Amélioration qualité (contraste, netteté, débruitage optionnel) | `enhance_image` | Non |
| 5 | Correction d'orientation franche (90/180/270°) | `detect_and_fix_orientation` (Partie 5) | Tesseract d'abord, modèle en repli |
| 6 | Redressement fin (inclinaison résiduelle) | `correct_skew` | Non |
| 7 | OCR complémentaire (diagnostic + indice pour le prompt) | `ocr_raw_text` | Non |

### Étape 0 — `extract_pdf_text_layer` *(nouveau, testé en premier)*
Avant tout traitement d'image, on vérifie si le PDF contient déjà une **couche texte native** (PDF généré
numériquement, ou déjà océrisé par un autre logiciel). Si oui, le texte est lu directement : instantané,
exact, et sans jamais solliciter la partie vision du modèle. Voir la cascade complète en Partie 6.

### Étape 1 — `pdf_to_images`
Convertit chaque page du PDF en image haute résolution. PyMuPDF applique déjà automatiquement la rotation
éventuellement déclarée dans les métadonnées de la page (`/Rotate`). Fonctionne identiquement pour un PDF
couleur ou un PDF issu d'un scan N&B : à ce stade, c'est juste une image.

### Étape 2 — `resize_for_model`
Plafonne la résolution avant tout envoi au modèle (voir sa docstring ci-dessous). **Point important suite
à votre dernier test** (`MAX_IMAGE_DIMENSION_MODEL=768`, bien en dessous du seuil habituellement rapporté,
et la réponse reste dégénérée) : ceci indique que la résolution n'est très probablement PAS (ou plus, si
`TRUST_REMOTE_CODE` a réglé le crash CUDA précédent) la cause principale chez vous. Le diagnostic croisé de
l'étape 7 et la Partie 5bis permettent de le confirmer plutôt que de le supposer.

### Étape 3 — `detect_color_mode` *(nouveau)*
Classe la page en « couleur » ou « noir et blanc » en mesurant la **fraction de pixels réellement
saturés** (pas une simple moyenne globale, biaisée par le fond blanc qui domine la plupart des documents
administratifs — un document réellement en couleur mais à fond blanc obtiendrait une moyenne trompeusement
basse). Sert de métadonnée de traçabilité dans les résultats (ex. repérer si l'extraction échoue plus
souvent sur les copies N&B) ; les étapes suivantes fonctionnent identiquement dans les deux cas.

### Étape 4 — `enhance_image`
Contraste local (CLAHE) + léger renforcement de netteté, systématiques. `ENABLE_DENOISING` (Partie 1,
désactivé par défaut) ajoute un débruitage (Non-Local Means) pour les scans VRAIMENT dégradés — plus lent
(CPU), à n'activer qu'en cas de besoin réel constaté.

### Étape 5 — `detect_and_fix_orientation` *(revue en Partie 5)*
Corrige une **rotation franche** (document scanné à l'envers ou sur le côté). Utilise maintenant
**Tesseract OSD en premier** (rapide, CPU, indépendant du modèle) ; ne se rabat sur le modèle Qwen3.5 que
si Tesseract est indisponible ou peu confiant sur cette page précise
(`OCR_ORIENTATION_CONFIDENCE_THRESHOLD`, Partie 1). Concrètement, la correction d'orientation fonctionne
maintenant même si le modèle a un problème.

### Étape 6 — `correct_skew`
Corrige une légère inclinaison résiduelle (quelques degrés), une fois le document déjà globalement à
l'endroit (étape 5) : sa détection d'angle suppose un texte à peu près horizontal.

### Étape 7 — `ocr_raw_text` *(nouveau)*
Extrait le texte brut de la page via Tesseract (langues : `OCR_LANGUAGES`, Partie 1 — français + arabe par
défaut). Utilisé de deux façons en Partie 6 : (a) comme **indice optionnel** ajouté au prompt d'extraction
pour aider le modèle sur les zones difficiles à lire ; (b) comme **signal de diagnostic** — si ce texte
est cohérent mais que le modèle renvoie une réponse dégénérée pour la même page, le message d'erreur le
signale explicitement au lieu de laisser croire à un problème de document.

*Validation* : les fonctions indépendantes du modèle ont été testées sur un document synthétique contenant
du **texte réellement lisible** (contrairement au tout premier jeu de test de ce projet, qui ne contenait
que des formes géométriques, insuffisant pour valider de l'OCR) : détection d'orientation correcte aux
4 angles, y compris après dégradation artificielle (flou + bruit + faible contraste), avec repli correct
vers une confiance basse quand l'image devient trop dégradée pour être fiable — voir le résumé de tests en
fin de notebook pour le détail.

In [ ]:
def pdf_to_images(pdf_path: Path, dpi: int = PDF_RENDER_DPI) -> list:
    """Convertit chaque page d'un PDF (y compris scanné) en une image PIL RGB haute résolution."""
    images = []
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(dpi=dpi)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            images.append(img)
    return images

In [ ]:
def extract_pdf_text_layer(pdf_path: Path, min_chars_per_page: int = TEXT_LAYER_MIN_CHARS) -> str:
    """Extrait la COUCHE TEXTE native du PDF, si elle existe (retourne "" sinon).

    Beaucoup de PDF "de scan" n'en sont pas vraiment : ils ont été produits numériquement, ou sont passés
    par un logiciel d'OCR qui a déposé une couche texte invisible par-dessus l'image. Dans ces cas, le
    texte est directement lisible SANS aucun modèle et SANS OCR -- c'est exact (pas d'erreur de
    reconnaissance), instantané, et totalement immunisé contre le problème de réponse dégénérée.
    C'est pourquoi on teste ce cas EN PREMIER, avant toute la chaîne coûteuse de traitement d'image.

    Le seuil min_chars_per_page évite de prendre pour une vraie couche texte les quelques caractères
    parasites (numéro de page, tampon, métadonnées du scanner) souvent présents même sur un vrai scan.
    """
    try:
        parts = []
        with pymupdf.open(pdf_path) as doc:
            for page in doc:
                parts.append(page.get_text("text") or "")
    except Exception as e:
        logger.warning("Lecture de la couche texte impossible pour %s (%s).", pdf_path.name, e)
        return ""

    if not parts:
        return ""
    total_chars = sum(len(p.strip()) for p in parts)
    if total_chars < min_chars_per_page * len(parts):
        return ""   # trop peu de texte -> c'est un vrai scan, il faudra passer par l'image
    return "\n\n".join(parts).strip()

In [ ]:
def resize_for_model(image: Image.Image, max_dimension: int = MAX_IMAGE_DIMENSION_MODEL) -> Image.Image:
    """Limite la plus grande dimension de l'image à max_dimension avant tout envoi au modèle.

    Des images de très haute résolution (un rendu PDF à 300 DPI dépasse largement max_dimension) ont été
    signalées par la communauté Hugging Face comme provoquant une réponse dégénérée du modèle Qwen3.5 sur
    les entrées image -- une suite de caractères répétés, ex. "!!!!!!!!!!!!!!!". Si le problème persiste
    même à une valeur basse de max_dimension (ex. 768px), ce n'est probablement plus la cause principale
    (voir le diagnostic croisé avec l'OCR, Partie 6, et la checklist de dépannage, Partie 5bis). Cette étape
    reste utile pour la vitesse : la résolution de l'image est directement liée au nombre de "tokens image"
    à traiter par le modèle. Ne redimensionne jamais vers le HAUT (une petite image reste inchangée).
    """
    w, h = image.size
    longest_side = max(w, h)
    if longest_side <= max_dimension:
        return image
    scale = max_dimension / longest_side
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    return image.resize(new_size, Image.LANCZOS)

In [ ]:
def detect_color_mode(image: Image.Image, pixel_saturation_threshold: int = 25,
                       color_fraction_threshold: float = 0.01) -> str:
    """Classe une page en 'couleur' ou 'noir_et_blanc'. Utilise la FRACTION de pixels dépassant un seuil de
    saturation (espace HSV), pas une simple moyenne globale : la plupart des documents administratifs sont
    dominés par un fond blanc peu saturé, ce qui biaiserait une moyenne même sur un document réellement en
    couleur (photo d'identité, cachet, logo). Validé : un document synthétique couleur (2,99% de pixels
    saturés) et sa version convertie en niveaux de gris (0%) sont correctement distingués avec ces seuils."""
    hsv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2HSV)
    fraction_colored = (hsv[:, :, 1] > pixel_saturation_threshold).mean()
    return "couleur" if fraction_colored > color_fraction_threshold else "noir_et_blanc"

In [ ]:
def correct_skew(image: Image.Image, max_angle: float = 15.0) -> Image.Image:
    """Corrige une légère inclinaison (quelques degrés) via une détection géométrique du contenu texte.
    Ignore volontairement les angles > max_angle : au-delà, il s'agit probablement d'une rotation franche
    (90/180/270°), gérée séparément par detect_and_fix_orientation (Partie 5)."""
    arr = np.array(image)
    gray = cv2.bitwise_not(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY))
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return image  # page quasi blanche : rien à corriger

    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) > max_angle or abs(angle) < 0.1:
        return image

    h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(arr, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rotated)

In [ ]:
def enhance_image(image: Image.Image) -> Image.Image:
    """Améliore un scan de mauvaise qualité : débruitage optionnel (ENABLE_DENOISING), puis contraste local
    (CLAHE sur le canal de luminance), puis léger renforcement de netteté (utile pour les scans flous).
    Volontairement PAS de binarisation : cela risquerait de dégrader la photo d'identité et les éléments
    colorés du document. Le résultat est explicitement re-borné à [0, 255] en uint8 avant conversion en
    image : un tableau hors de cette plage (dépassement possible après le renforcement de netteté) donnerait
    une image corrompue une fois envoyée au modèle."""
    arr = np.array(image)

    if ENABLE_DENOISING:
        # Non-Local Means : efficace mais nettement plus lent que le reste de cette fonction -- réservé aux
        # scans vraiment très dégradés (bruit de fond marqué, grain de scanner).
        arr = cv2.fastNlMeansDenoisingColored(arr, None, h=7, hColor=7, templateWindowSize=7, searchWindowSize=21)

    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    result = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

    # Renforcement léger de netteté (unsharp mask) : aide les scans flous sans créer d'artefacts visibles
    # si le scan était déjà net.
    blurred = cv2.GaussianBlur(result, (0, 0), sigmaX=2.0)
    sharpened = cv2.addWeighted(result, 1.5, blurred, -0.5, 0)
    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    return Image.fromarray(sharpened)

In [ ]:
def ocr_raw_text(image: Image.Image, languages: str = OCR_LANGUAGES) -> str:
    """Extrait le texte brut d'une page via Tesseract. Ne lève JAMAIS d'exception : retourne une chaîne
    vide si Tesseract est indisponible ou échoue -- cette fonction est une AIDE (indice pour le prompt +
    diagnostic), pas un point de défaillance critique du pipeline."""
    if not (ENABLE_OCR_ASSIST and TESSERACT_AVAILABLE):
        return ""
    try:
        return pytesseract.image_to_string(image, lang=languages).strip()
    except Exception as e:
        logger.warning("OCR Tesseract impossible sur cette page (%s).", e)
        return ""

In [ ]:
def detect_orientation_tesseract(image: Image.Image):
    """Détecte une rotation franche (0/90/180/270°) via Tesseract OSD (Orientation and Script Detection) --
    rapide, CPU, INDÉPENDANT du modèle Qwen3.5. Retourne None (pas d'avis fiable) si Tesseract est
    indisponible, échoue, ou si sa confiance est sous OCR_ORIENTATION_CONFIDENCE_THRESHOLD (Partie 1) ; dans
    ce cas l'appelant (detect_and_fix_orientation, Partie 5) se rabat sur le modèle. Validé sur un document
    de test avec texte réel : détection correcte aux 4 angles avec une confiance ~3.5-4.0 sur image propre,
    ~2.8 sur image dégradée (toujours correcte), et repli correct (confiance ~1.1, sous le seuil) sur une
    image à la fois dégradée ET mal orientée où la détection géométrique devient réellement peu fiable."""
    if not (ENABLE_OCR_ASSIST and TESSERACT_AVAILABLE):
        return None
    try:
        osd = pytesseract.image_to_osd(image, output_type=pytesseract.Output.DICT)
    except Exception as e:
        logger.debug("Tesseract OSD indisponible sur cette page (%s) -- repli sur le modèle.", e)
        return None
    if osd.get("orientation_conf", 0) < OCR_ORIENTATION_CONFIDENCE_THRESHOLD:
        return None
    return int(osd["rotate"])  # degrés à tourner en sens horaire pour corriger -- même convention que le modèle

## Partie 5 — Chargement du modèle Qwen3.5 (vision-langage) en local

Le `config.json` fourni indique `"model_type": "qwen3_5"` : il s'agit de l'architecture **Qwen3.5**, un
modèle nativement multimodal (texte / image / vidéo), supporté par `transformers` à partir de la version
5.8. Le checkpoint est quantifié **FP8** (probablement au format `compressed-tensors`, d'où sa dépendance
installée en Partie 0) : la cellule suivante inspecte le `quantization_config` réel pour confirmer le
format avant chargement.

On charge le modèle via **`AutoModelForImageTextToText`** (résout automatiquement la classe
`Qwen3_5ForConditionalGeneration` à partir de `config.json` — approche recommandée par la fiche officielle
du modèle plutôt que d'importer la classe spécifique en dur), avec `dtype="auto"` (respecte la précision
déjà présente dans le checkpoint, donc le FP8) et `device_map="auto"` (répartition automatique sur le(s)
GPU disponibles). On tente `attn_implementation="sdpa"` (backend d'attention natif PyTorch, rapide, sans
dépendance supplémentaire) avec repli automatique si votre environnement ne le supporte pas.

**Important pour un checkpoint FP8** : `trust_remote_code` (`TRUST_REMOTE_CODE`, Partie 1) conditionne le
téléchargement du noyau de calcul optimisé pour le FP8 — sans lui, transformers bascule sur une
implémentation de repli qui peut être instable (voir Partie 5bis en cas d'erreur CUDA).

In [ ]:
config_path = Path(MODEL_PATH) / "config.json"
if not config_path.exists():
    raise FileNotFoundError(f"config.json introuvable à {config_path} — vérifiez MODEL_PATH (Partie 1).")

with open(config_path, encoding="utf-8") as f:
    model_config = json.load(f)

print("model_type            :", model_config.get("model_type"))
print("transformers_version  :", model_config.get("transformers_version"), "(version utilisée lors de la sauvegarde du modèle)")
print("quantization_config   :")
print(json.dumps(model_config.get("quantization_config", {}), indent=2, ensure_ascii=False))

quant_method = str(model_config.get("quantization_config", {}).get("quant_method", "")).lower()
if "compressed" in quant_method or "fp8" in quant_method:
    print(
        "\nCheckpoint FP8 (compressed-tensors) détecté. Pour ce type de checkpoint, transformers essaie "
        "d'utiliser un noyau de calcul optimisé (DeepGEMM/CUTLASS, souvent récupéré depuis le Hub via la "
        "librairie `kernels`) ; s'il ne peut pas être chargé (GPU non compatible, ou dépôt non reconnu comme "
        "'éditeur de confiance' -- voir TRUST_REMOTE_CODE, Partie 1), transformers bascule sur une "
        "implémentation Triton plus lente et, sur certains environnements, instable. Si le chargement du "
        "modèle ci-dessous affiche un avertissement 'CUTLASS quantization kernel' / 'publisher trust status', "
        "voir Partie 5bis en cas d'erreur CUDA pendant la génération."
    )

In [ ]:
t0 = time.time()
logger.info("Chargement du modèle depuis %s ...", MODEL_PATH)

# Si cette ligne échoue avec une erreur "unrecognized model type" pour "qwen3_5", votre version de
# transformers est probablement antérieure à la 5.8 : exécutez `%pip install -U transformers` puis
# redémarrez le kernel.
#
# ⚠️ Si vous avez DÉJÀ vu une erreur CUDA (AcceleratorError / cudaErrorUnknown / RuntimeError CUDA...) dans
# cette session, REDÉMARREZ LE KERNEL avant de relancer cette cellule : un contexte CUDA corrompu par une
# erreur ne se répare pas en réexécutant simplement le code, y compris avec les correctifs ci-dessous.
#
# attn_implementation="sdpa" accélère les couches d'attention "classique" (backend natif PyTorch, aucune
# dépendance supplémentaire). On retente sans ce réglage si votre environnement ne le supporte pas pour
# cette architecture, plutôt que de faire échouer tout le chargement pour un gain de vitesse secondaire.
#
# trust_remote_code=TRUST_REMOTE_CODE (Partie 1) autorise le téléchargement du noyau FP8 optimisé
# (voir la cellule précédente) plutôt que la bascule vers l'implémentation de repli Triton.
#
# MODEL_LOAD_MODE (Partie 1) choisit la précision de calcul. "bf16" (défaut) déquantifie le checkpoint et
# contourne entièrement le chemin FP8 -- c'est le réglage qui résout le symptôme "!!!!" lorsqu'il provient
# d'un NaN dans le noyau quantifié.
_DTYPE_BY_MODE = {
    "bf16": torch.bfloat16,   # contourne le calcul FP8 (recommandé si réponses dégénérées / crash CUDA)
    "fp16": torch.float16,    # déconseillé sur Qwen-VL : cause connue de NaN
    "fp8": "auto",            # conserve la quantification du checkpoint
}
if MODEL_LOAD_MODE not in _DTYPE_BY_MODE:
    raise ValueError(f"MODEL_LOAD_MODE invalide : {MODEL_LOAD_MODE!r} (attendu : {list(_DTYPE_BY_MODE)})")
_dtype = _DTYPE_BY_MODE[MODEL_LOAD_MODE]
logger.info("Mode de chargement : %s (dtype=%s)", MODEL_LOAD_MODE, _dtype)
if MODEL_LOAD_MODE == "fp16":
    logger.warning(
        "MODEL_LOAD_MODE='fp16' : float16 est une cause connue de NaN (réponses '!!!!') sur les modèles "
        "Qwen-VL, entraînés en bfloat16. Préférez 'bf16' si votre GPU le permet."
    )

_load_kwargs = dict(dtype=_dtype, device_map="auto", trust_remote_code=TRUST_REMOTE_CODE)
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH, attn_implementation="sdpa", **_load_kwargs
    )
except (ValueError, TypeError) as e:
    logger.warning("attn_implementation='sdpa' indisponible (%s) — nouvelle tentative sans ce réglage.", e)
    model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **_load_kwargs)

processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=TRUST_REMOTE_CODE)
model.eval()

logger.info("Modèle chargé en %.1fs.", time.time() - t0)
if torch.cuda.is_available():
    print(f"Mémoire GPU allouée après chargement : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

# --- Diagnostic important pour la VITESSE (Partie 7) ---
# Un modèle de cette taille doit tenir ENTIÈREMENT sur GPU. Si device_map="auto" a dû répartir ne serait-ce
# qu'une couche sur le CPU (mémoire GPU insuffisante), l'inférence devient extrêmement lente (facteur
# 10x-100x, largement plus déterminant que tout autre réglage de ce notebook). Vérifiez ci-dessous
# qu'aucun appareil autre que "cuda:N" n'apparaît.
device_map = getattr(model, "hf_device_map", None)
if device_map:
    devices_used = sorted(set(str(d) for d in device_map.values()))
    print("Répartition du modèle sur les appareils :", devices_used)
    if any(("cpu" in d or "disk" in d) for d in devices_used):
        print(
            "\n⚠️  ATTENTION : une partie du modèle est déchargée sur CPU/disque. C'est très probablement "
            "la cause principale d'un traitement extrêmement lent en Partie 7, avant toute autre "
            "optimisation. Pistes : libérer de la mémoire GPU (autres processus), demander un GPU avec "
            "plus de VRAM, ou utiliser une variante du modèle plus petite/plus quantifiée."
        )
else:
    print("Modèle chargé sans device_map détaillé (mono-GPU ou CPU uniquement probable).")

Fonction générique d'appel au modèle, et détection/correction des rotations franches (90/180/270°) —
cette dernière a besoin du modèle chargé ci-dessus, d'où sa place ici plutôt qu'en Partie 4.

In [ ]:
def _ask_model(images: list, prompt: str, max_new_tokens: int) -> str:
    """Appel générique du modèle avec une ou plusieurs images + une consigne texte. Retourne la réponse brute."""
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,   # garde-fou contre les boucles de répétition (ex. "!!!!!!!!!!!!!!!")
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
        )

    trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()


def _looks_degenerate(text: str) -> bool:
    """Détecte une réponse manifestement invalide du modèle (vide, ou dominée par un seul caractère
    répété, ex. "!!!!!!!!!!!!!!!") plutôt qu'une vraie réponse à interpréter. Un tel résultat indique
    presque toujours un problème en amont (image trop volumineuse, problème de génération) -- voir la
    checklist de dépannage en Partie 5bis -- plutôt qu'une réponse du modèle qu'il suffirait de reparser."""
    stripped = text.strip()
    if not stripped:
        return True
    if len(stripped) == 1:
        return not stripped.isalnum()  # un seul caractère : valide seulement s'il est alphanumérique
                                        # (ex. "0" est une vraie réponse d'orientation, "!" ne l'est pas)
    most_common_count = max(stripped.count(c) for c in set(stripped))
    return most_common_count / len(stripped) > 0.8


ORIENTATION_PROMPT = (
    "Regarde cette image de document scanné. Le texte est-il actuellement à l'endroit et horizontal ? "
    "Sinon, de combien de degrés faut-il la faire pivoter DANS LE SENS DES AIGUILLES D'UNE MONTRE pour "
    "qu'il le devienne : 90, 180 ou 270 ? Réponds uniquement par un seul chiffre parmi 0, 90, 180, 270 "
    "— aucun autre mot."
)

def detect_and_fix_orientation(image: Image.Image) -> Image.Image:
    """Détecte une rotation franche (90/180/270°) et corrige l'image en conséquence.

    Essaie D'ABORD Tesseract OSD (detect_orientation_tesseract, Partie 4) : rapide, CPU, indépendant du
    modèle. Ne se rabat sur le modèle Qwen3.5 que si Tesseract est indisponible ou pas assez confiant sur
    cette page précise. Journalise explicitement (WARNING) toute réponse dégénérée ou non reconnue du
    modèle plutôt que de la traiter silencieusement comme "0° - aucune rotation nécessaire" : une réponse
    dégénérée ici est un signal d'alerte général sur le pipeline, pas une simple absence de rotation."""
    if not ENABLE_ORIENTATION_CHECK:
        return image

    rotation_needed = detect_orientation_tesseract(image)
    source = "Tesseract OSD"

    if rotation_needed is None:
        source = "modèle"
        try:
            answer = _ask_model([image], ORIENTATION_PROMPT, max_new_tokens=8)
        except Exception as e:
            logger.warning("Détection d'orientation impossible (%s) — image conservée telle quelle.", e)
            return image

        if _looks_degenerate(answer):
            logger.warning(
                "Réponse dégénérée du modèle pendant la détection d'orientation (%r) — probable symptôme "
                "d'un problème en amont (voir la checklist de dépannage, Partie 5bis), pas un vrai résultat "
                "d'orientation. Image conservée telle quelle.",
                answer[:30],
            )
            return image

        match = re.search(r"\b(90|180|270|0)\b", answer)
        if not match:
            logger.warning(
                "Réponse inattendue du modèle pendant la détection d'orientation (%r) — aucune rotation "
                "reconnue, image conservée telle quelle.",
                answer[:60],
            )
            return image
        rotation_needed = int(match.group(1))

    logger.debug("Orientation détectée via %s : %d°", source, rotation_needed)
    if rotation_needed == 0:
        return image
    # PIL.Image.rotate() tourne dans le sens ANTIhoraire pour un angle positif -> signe négatif pour un pivot horaire
    return image.rotate(-rotation_needed, expand=True)

`preprocess_page` assemble les étapes de la Partie 4 + `detect_and_fix_orientation` ci-dessus, dans
l'ordre validé, et est utilisée aussi bien par le test de sanité (Partie 5bis) que par le traitement par
lot (Partie 6/7) : l'aperçu affiché en Partie 5bis correspond ainsi EXACTEMENT à ce que le modèle reçoit
en production — les deux ne peuvent plus diverger silencieusement.

In [ ]:
def preprocess_page(image: Image.Image) -> Image.Image:
    """Chaîne de prétraitement complète d'une page, dans l'ordre validé (voir Partie 4 pour le détail de
    chaque étape) :
    1. resize_for_model            — sécurité/vitesse, en tout premier
    2. enhance_image                — contraste + netteté (+ débruitage optionnel), avant toute décision
                                       géométrique ou modèle
    3. detect_and_fix_orientation   — rotation franche (90/180/270°) : Tesseract d'abord, modèle en repli
    4. correct_skew                 — inclinaison résiduelle, une fois le document déjà à l'endroit
    (detect_color_mode et ocr_raw_text, Partie 4, sont appelées séparément par process_one_client, Partie 6 :
    ce sont des métadonnées/diagnostics, pas des transformations de l'image elle-même.)
    """
    img = resize_for_model(image)
    img = enhance_image(img) if ENABLE_CONTRAST_ENHANCEMENT else img
    img = detect_and_fix_orientation(img) if ENABLE_ORIENTATION_CHECK else img
    img = correct_skew(img) if ENABLE_SKEW_CORRECTION else img
    return img

## Partie 5bis-A — Escalier de diagnostic (à exécuter en premier)

Vous avez déjà essayé de résoudre le `!!!!!!!!` en changeant la résolution d'image (`768`) et le nombre de
tokens (`10000`), sans effet. Plutôt que de continuer à faire varier des paramètres au hasard, cette
section **isole la cause** en quatre tests de complexité croissante. Le principe : chaque niveau ajoute une
seule brique. **Le premier niveau qui échoue désigne le coupable.**

| Niveau | Ce qu'on teste | Si ça échoue à ce niveau, la cause est… |
|---|---|---|
| 0 | Un simple passage avant, sans génération : les logits contiennent-ils des `NaN` ? | Le **calcul du modèle** (quantification/précision). Rien à voir avec vos documents. |
| 1 | Génération à partir de **texte seul**, aucune image | Le **modèle de langage** lui-même (précision, poids, kernel) |
| 2 | Génération sur une **image synthétique parfaitement lisible** (générée ici, pas un scan) | La **partie vision** ou le processeur d'images |
| 3 | Génération sur **votre vrai scan** (Partie 5bis-B ci-dessous) | Vos **documents** (qualité, orientation, résolution) |

Concrètement : si le niveau 0 ou 1 échoue, **aucune amélioration du prétraitement d'image ne peut vous
aider** — c'est le chemin de calcul qu'il faut corriger (voir `MODEL_LOAD_MODE`, Partie 1). C'est
l'hypothèse la plus probable dans votre cas, cohérente à la fois avec le crash CUDA dans
`w8a8_block_fp8_matmul` et avec le fait que réduire la résolution à 768 n'a rien changé.

In [ ]:
diagnostic_results = {}

# --- Niveau 0 : les logits contiennent-ils des NaN ? (test décisif, sans génération) ---
# C'est LE test qui tranche : une réponse "!!!!" vient presque toujours de logits NaN. Quand tous les logits
# valent NaN, argmax renvoie systématiquement le token 0 (= "!" dans le vocabulaire Qwen), et le modèle
# "écrit" donc des "!" jusqu'à épuisement de max_new_tokens.
try:
    _test_inputs = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": "Bonjour"}]}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        _logits = model(**_test_inputs).logits

    n_nan = torch.isnan(_logits).sum().item()
    n_inf = torch.isinf(_logits).sum().item()
    diagnostic_results["niveau_0_logits_finis"] = (n_nan == 0 and n_inf == 0)

    print(f"[Niveau 0] logits : {_logits.numel()} valeurs, {n_nan} NaN, {n_inf} Inf")
    if n_nan or n_inf:
        print(
            "  ❌ ÉCHEC — le calcul du modèle produit des valeurs invalides AVANT même toute génération et\n"
            "     SANS AUCUNE IMAGE. C'est la confirmation directe que le problème vient du chemin de calcul\n"
            "     (quantification FP8 / précision), et NON de vos documents.\n"
            "     -> Correctif : MODEL_LOAD_MODE = 'bf16' (Partie 1), puis REDÉMARREZ LE KERNEL et relancez."
        )
    else:
        print("  ✅ OK — les logits sont numériquement sains (aucun NaN/Inf).")
except Exception as e:
    diagnostic_results["niveau_0_logits_finis"] = None
    print(f"[Niveau 0] Test impossible ({type(e).__name__}: {e})")

In [ ]:
# --- Niveau 1 : génération à partir de texte seul, sans aucune image ---
try:
    answer_text_only = _ask_model([], "Réponds simplement par : bonjour", max_new_tokens=20)
    ok_1 = not _looks_degenerate(answer_text_only)
    diagnostic_results["niveau_1_texte_seul"] = ok_1
    print(f"[Niveau 1] Réponse texte seul : {answer_text_only!r}")
    print("  ✅ OK — le modèle de langage génère normalement." if ok_1 else
          "  ❌ ÉCHEC — dégénéré SANS AUCUNE IMAGE : le problème ne vient pas de vos documents.\n"
          "     -> Correctif : MODEL_LOAD_MODE = 'bf16' (Partie 1), redémarrage du kernel, puis relancer.")
except Exception as e:
    diagnostic_results["niveau_1_texte_seul"] = None
    print(f"[Niveau 1] Test impossible ({type(e).__name__}: {e})")

In [ ]:
# --- Niveau 2 : image synthétique PARFAITEMENT lisible (générée ici, aucun scan impliqué) ---
# Si ce niveau échoue alors que le niveau 1 passe, le problème est dans la partie vision / le processeur
# d'images -- et non dans la qualité de vos scans, puisque cette image-ci est nette par construction.
try:
    _synthetic = Image.new("RGB", (600, 200), "white")
    _draw = ImageDraw.Draw(_synthetic)
    _draw.text((40, 80), "CARTE NATIONALE 12345", fill="black")
    display(_synthetic)

    answer_synthetic = _ask_model([_synthetic], "Transcris le texte visible sur cette image.", max_new_tokens=50)
    ok_2 = not _looks_degenerate(answer_synthetic)
    diagnostic_results["niveau_2_image_synthetique"] = ok_2
    print(f"[Niveau 2] Réponse sur image parfaite : {answer_synthetic!r}")
    if ok_2:
        print("  ✅ OK — la partie vision fonctionne. Si le niveau 3 échoue, ce sont bien vos documents.")
    else:
        print(
            "  ❌ ÉCHEC sur une image pourtant parfaitement nette : le problème vient du traitement des\n"
            "     images par le modèle, PAS de la qualité de vos scans. Améliorer le prétraitement serait\n"
            "     inutile ici. -> Voir MODEL_LOAD_MODE (Partie 1) et la checklist en fin de Partie 5bis-B."
        )
except Exception as e:
    diagnostic_results["niveau_2_image_synthetique"] = None
    print(f"[Niveau 2] Test impossible ({type(e).__name__}: {e})")

In [ ]:
# --- Synthèse de l'escalier de diagnostic ---
print("Résultats :", json.dumps(diagnostic_results, ensure_ascii=False))
if diagnostic_results.get("niveau_0_logits_finis") is False or diagnostic_results.get("niveau_1_texte_seul") is False:
    print(
        "\n>>> CONCLUSION : le problème est dans le CHEMIN DE CALCUL du modèle, pas dans vos documents.\n"
        ">>> Action : MODEL_LOAD_MODE = 'bf16' en Partie 1, PUIS redémarrer le kernel (obligatoire), puis\n"
        ">>> réexécuter le notebook depuis le début. Inutile de toucher au prétraitement d'image."
    )
elif diagnostic_results.get("niveau_2_image_synthetique") is False:
    print("\n>>> CONCLUSION : partie vision/processeur en cause — voir la checklist en fin de Partie 5bis-B.")
elif all(v for v in diagnostic_results.values() if v is not None):
    print("\n>>> Le modèle est sain. Si le test sur vrai scan échoue ci-dessous, la cause est dans les documents.")

## Partie 5bis-B — Test de sanité sur un vrai document

Ce niveau 3 vérifie sur **un seul client** que la chaîne complète fonctionne de bout en bout. Cela évite de
découvrir un problème seulement après avoir attendu la fin d'un traitement par lot de plusieurs dizaines de
minutes. L'image affichée ci-dessous passe par `preprocess_page` — exactement la même chaîne que celle
utilisée en Partie 7 — donc ce que vous voyez ici est bien ce que le modèle reçoit en production.

In [ ]:
if not target_clients:
    print("Aucun client cible — vérifiez le rapport d'inventaire (Partie 2) avant de continuer.")
else:
    sample_client = target_clients[0]
    sample_pages = pdf_to_images(FILTERED_DIR / sample_client / IDENTITY_FILE)
    print(f"Client de test : {sample_client} — {len(sample_pages)} page(s) détectée(s).")

    sample_page = preprocess_page(sample_pages[0])
    print("Mode couleur détecté :", detect_color_mode(sample_page))
    display(sample_page)  # aperçu de la page telle que le modèle va la recevoir

    # Diagnostic croisé, INDÉPENDANT du modèle : si Tesseract lit un texte cohérent sur cette image, on saura
    # que l'image elle-même est exploitable -- utile pour interpréter la réponse du modèle ci-dessous.
    ocr_preview = ocr_raw_text(sample_page)
    if ENABLE_OCR_ASSIST and TESSERACT_AVAILABLE:
        print(f"\nAperçu OCR Tesseract ({len(ocr_preview)} caractère(s) extraits) :")
        print(ocr_preview[:300] if ocr_preview else "(rien d'extrait)")
    else:
        print("\n(OCR Tesseract désactivé ou indisponible -- voir ENABLE_OCR_ASSIST, Partie 1)")

    quick_answer = _ask_model(
        [sample_page],
        "En une phrase : quel type de document est visible sur cette image, et le texte te semble-t-il "
        "net et bien orienté ?",
        max_new_tokens=100,
    )
    print("\nRéponse du modèle :", quick_answer)

    if _looks_degenerate(quick_answer):
        if len(ocr_preview) > 20:
            print(
                "\n⚠️  Réponse du modèle DÉGÉNÉRÉE (caractère répété), MAIS Tesseract a bien extrait du "
                "texte cohérent sur cette même image (aperçu ci-dessus). Ceci pointe vers un problème côté "
                "MODÈLE/ENVIRONNEMENT (kernel FP8, CUDA...), PAS côté document -- voir Cas 1 de la "
                "checklist juste en dessous."
            )
        else:
            print(
                "\n⚠️  Réponse du modèle DÉGÉNÉRÉE, et Tesseract n'a lui non plus rien extrait de "
                "cohérent -- l'image elle-même pourrait être en cause (ou Tesseract est indisponible : "
                "vérifiez TESSERACT_AVAILABLE ci-dessus). Voir la checklist juste en dessous."
            )
    else:
        print("\nSi cette réponse est cohérente, vous pouvez passer à la suite en toute confiance.")

### Si la réponse ci-dessus est illisible (ex. une suite de `!`), ou si une erreur CUDA apparaît

**D'abord, une règle absolue : après toute erreur CUDA (`AcceleratorError`, `cudaErrorUnknown`,
`RuntimeError` mentionnant CUDA...), REDÉMARREZ LE KERNEL avant de retenter quoi que ce soit.** Une fois
le contexte CUDA corrompu par une erreur, tous les appels GPU suivants échouent de la même façon dans le
même processus — réexécuter une cellule, même corrigée, ne suffit pas.

**Cas 1 — une vraie erreur Python/CUDA apparaît, OU l'aperçu OCR ci-dessus montre du texte cohérent alors
que le modèle reste dégénéré** (`AcceleratorError`, trace passant par `w8a8_block_fp8_matmul` /
`finegrained_fp8.py`, souvent précédée d'un avertissement "Failed to load CUTLASS quantization kernel" /
"could not verify publisher trust status") : c'est le noyau de calcul FP8 optimisé qui n'a pas pu être
chargé (voir Partie 1, `TRUST_REMOTE_CODE`, et l'explication juste après l'inspection du
`quantization_config` en Partie 5), forçant une implémentation de repli instable sur cet environnement —
**ni la résolution d'image, ni la qualité du scan n'y changeront rien**. Dans l'ordre, **après avoir
redémarré le kernel** :
1. **`MODEL_LOAD_MODE = "bf16"` (Partie 1)** — le correctif principal : il contourne entièrement le calcul
   FP8. À faire en priorité si le niveau 0 ou 1 de l'escalier de diagnostic a échoué.
2. **Redémarrez le kernel**, puis réexécutez le notebook depuis le début (non négociable après une erreur
   CUDA, et nécessaire de toute façon pour recharger le modèle avec la nouvelle précision).
3. Si vous préférez conserver le FP8 pour économiser la mémoire GPU : vérifiez `TRUST_REMOTE_CODE = True`
   et l'absence d'avertissement CUTLASS au chargement, puis relancez l'escalier de diagnostic pour
   confirmer que les logits sont redevenus sains avant de lancer un lot complet.
4. Vérifiez `pip show kernels compressed-tensors transformers` : ce sont des librairies très récentes,
   une mise à jour peut suffire.
5. Ne comptez pas sur `MAX_NEW_TOKENS_EXTRACTION` : l'augmenter (ex. 10000) allonge seulement la suite de
   `!` et ralentit le traitement. Laissez-le à 2048.

**Cas 2 — pas d'erreur Python, réponse dégénérée, ET Tesseract n'extrait rien de cohérent non plus** :
l'image elle-même est probablement en cause (résolution, qualité). Réduisez `MAX_IMAGE_DIMENSION_MODEL`
(Partie 1), activez `ENABLE_DENOISING` si le scan est très bruité, et vérifiez le diagnostic GPU/CPU
(Partie 5). Si vous avez déjà testé `MAX_IMAGE_DIMENSION_MODEL=768` sans succès (comme rapporté), ce cas
devient nettement moins probable que le Cas 1 — revérifiez d'abord `TRUST_REMOTE_CODE`.

Cette checklist est fondée sur des rapports communautaires (Hugging Face) et sur les traces d'erreur que
vous avez rencontrées, pas sur une documentation officielle Qwen/Alibaba — à ajuster si vous identifiez
une autre cause dans votre environnement.

## Partie 6 — Extraction structurée : une stratégie en cascade

Plutôt qu'un appel unique au modèle (fragile : s'il échoue, tout est perdu), l'extraction suit maintenant
**quatre étapes en cascade**. Chaque étape n'est tentée que si la précédente n'a pas suffi, du moins
coûteux/plus fiable vers le plus coûteux :

| Étape | Technique | Quand elle s'applique | Coût |
|---|---|---|---|
| **A** | Lecture de la **couche texte native** du PDF (`extract_pdf_text_layer`) | PDF produit numériquement, ou déjà passé par un logiciel d'OCR | Quasi nul, et **exact** |
| **B** | Rendu image + **prétraitement complet** (les 7 étapes de la Partie 4) | Vrai scan, sans couche texte | CPU |
| **C** | Appel(s) au **modèle**, par groupes de pages, avec nouvelles tentatives | Toujours, sur un vrai scan | GPU |
| **D** | **Repli OCR seul** (texte brut conservé) | Le modèle a définitivement échoué | CPU |

### Étape A — la couche texte d'abord
Beaucoup de PDF « de scan » n'en sont pas vraiment : ils ont été générés numériquement, ou un logiciel
d'OCR y a déposé une couche texte invisible. Dans ce cas le texte est directement lisible, **sans erreur
de reconnaissance, instantanément, et sans jamais solliciter la partie vision du modèle** — donc immunisé
contre le problème de réponse dégénérée. C'est pourquoi ce test passe en premier. `TEXT_LAYER_MIN_CHARS`
(Partie 1) évite de confondre une vraie couche texte avec les quelques caractères parasites qu'un scanner
dépose parfois (numéro de page, tampon).

### Étape C — pourquoi découper en groupes de pages
Envoyer *toutes* les pages en un seul appel permet au modèle de croiser recto et verso, mais ne passe pas
à l'échelle : au-delà de quelques images, la mémoire GPU et la latence explosent, et la qualité des
réponses se dégrade. Le document est donc découpé en groupes d'au plus `MAX_PAGES_PER_CALL` pages
(Partie 1, 4 par défaut — un recto/verso tient donc toujours dans un seul appel), puis les résultats sont
fusionnés par `merge_extractions`.

**Règle de fusion, volontairement prudente** : on garde la première valeur non vide ; et si un groupe
ultérieur propose une valeur *différente* pour un champ déjà rempli, la valeur n'est pas écrasée
silencieusement — la divergence est consignée dans `champs_divergents` pour revue manuelle. En KYC, une
divergence sur un numéro de document ou une date d'expiration doit être vue par un humain, pas arbitrée
automatiquement par le pipeline.

### Escalier de nouvelles tentatives (`_extract_structured_with_retries`)
En cas de réponse dégénérée ou de JSON illisible, l'appel est retenté aux résolutions plus basses de
`RETRY_RESOLUTIONS` (Partie 1). La résolution est la seule variable qu'il soit utile de faire varier
automatiquement : les autres causes possibles (chemin de calcul FP8, précision) exigent un rechargement
du modèle et ne peuvent pas être corrigées en cours d'exécution — d'où `MODEL_LOAD_MODE` et l'escalier de
diagnostic de la Partie 5bis-A.

### Étape D — ne jamais repartir les mains vides
Si le modèle échoue définitivement mais que Tesseract a lu du texte cohérent, le client n'est pas marqué
en échec sec : le statut devient `succes_partiel`, le **texte OCR brut est conservé** dans le résultat
(permettant une saisie ou vérification manuelle), et le message d'erreur indique explicitement que la
cause est côté modèle/environnement et non côté document — puisque l'OCR, lui, a réussi sur cette même
image.

### Le prompt
- couvre les documents multilingues (français / arabe / anglais / tamazight / mélange), fréquents sur les
  pièces d'identité algériennes souvent bilingues ;
- demande une sortie **JSON strict**, avec un champ `texte_brut_ocr` (transcription complète, en
  repli/traçabilité) et un champ `champs_incertains` (liste des champs à faible confiance, pour cibler la
  revue manuelle) ;
- interdit explicitement au modèle de traduire les noms propres ou d'inventer une valeur absente ;
- reçoit, si disponible, un **indice OCR** (texte brut extrait par Tesseract, Partie 4) en complément de
  l'image — présenté explicitement comme pouvant contenir des erreurs, à utiliser seulement pour aider sur
  les zones difficiles à lire, jamais comme vérité absolue.

Adaptez librement le schéma ci-dessous aux champs réellement exigés par votre équipe conformité.

In [ ]:
EXTRACTION_PROMPT = """Tu es un système expert en extraction de données à partir de documents d'identité (carte nationale d'identité, passeport, permis de conduire, titre de séjour, etc.) dans un contexte bancaire de connaissance client (KYC).

Les images fournies sont TOUTES les pages d'un même document, appartenant à un même client (par exemple le recto et le verso d'une carte d'identité). Le document peut être rédigé en français, en arabe, en anglais, en tamazight, ou dans un mélange de ces langues (les cartes d'identité algériennes sont souvent bilingues arabe/français). Le scan peut être de mauvaise qualité, incliné, flou, ou comporter de l'écriture manuscrite.

Analyse l'ensemble des pages fournies et réponds UNIQUEMENT avec un objet JSON valide (sans texte avant ou après, sans balises markdown), respectant exactement ce schéma :

{
  "type_document": string ou null,
  "nom": string ou null,
  "prenom": string ou null,
  "nom_arabe": string ou null,
  "prenom_arabe": string ou null,
  "date_naissance": string ou null,
  "lieu_naissance": string ou null,
  "sexe": string ou null,
  "nationalite": string ou null,
  "numero_document": string ou null,
  "numero_identification_nationale": string ou null,
  "date_delivrance": string ou null,
  "date_expiration": string ou null,
  "autorite_delivrance": string ou null,
  "adresse": string ou null,
  "langues_detectees": string ou null,
  "champs_incertains": [string],
  "texte_brut_ocr": string
}

Règles impératives :
- Ne traduis JAMAIS les noms propres, adresses ou numéros : recopie-les exactement comme ils apparaissent.
- Si une information est absente ou illisible, mets null plutôt que d'inventer une valeur.
- Liste dans "champs_incertains" le nom de chaque champ que tu n'es pas sûr d'avoir correctement lu.
- "texte_brut_ocr" doit contenir une transcription complète de tout le texte visible, page par page.
"""

OCR_HINT_TEMPLATE = """

Indice supplémentaire (extraction automatique par OCR, peut contenir des erreurs -- ne t'y fie pas
aveuglément, utilise-le seulement pour t'aider à lire les zones difficiles de l'image) :
---
{ocr_text}
---
"""

def build_extraction_prompt(ocr_hint_text: str = "") -> str:
    """Construit le prompt final envoyé au modèle, en ajoutant l'indice OCR (Partie 4, ocr_raw_text) s'il
    est disponible et suffisamment substantiel pour être utile (évite d'ajouter du bruit si l'OCR n'a
    presque rien extrait)."""
    if ocr_hint_text and len(ocr_hint_text) > 20:
        return EXTRACTION_PROMPT + OCR_HINT_TEMPLATE.format(ocr_text=ocr_hint_text[:4000])
    return EXTRACTION_PROMPT

print(f"Longueur du prompt d'extraction (sans indice OCR) : {len(EXTRACTION_PROMPT)} caractères.")

In [ ]:
def extract_json_from_text(text: str) -> dict:
    """Extrait le premier objet JSON valide d'une réponse de modèle, même si celle-ci contient des balises
    ```json ... ``` ou du texte superflu avant/après malgré la consigne."""
    cleaned = re.sub(r"^```(?:json)?", "", text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    if start == -1:
        raise ValueError("Aucun objet JSON trouvé dans la réponse du modèle.")
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(cleaned[start:i + 1])
    raise ValueError("Objet JSON incomplet dans la réponse du modèle.")

In [ ]:
def merge_extractions(partials: list) -> dict:
    """Fusionne les résultats JSON de plusieurs appels (document découpé en groupes de pages).

    Règle de fusion, volontairement conservatrice pour un usage KYC :
    - pour un champ simple, on garde la PREMIÈRE valeur non vide rencontrée (les pages sont traitées dans
      l'ordre, et le recto d'une pièce d'identité porte l'essentiel des informations) ;
    - une valeur déjà remplie n'est JAMAIS écrasée silencieusement : si un groupe de pages ultérieur
      propose une valeur différente, la divergence est consignée dans "champs_divergents" pour revue
      manuelle, plutôt que de choisir arbitrairement (une divergence sur un n° de document ou une date
      d'expiration doit être vue par un humain, pas arbitrée par le pipeline) ;
    - les listes (ex. langues_detectees, champs_incertains) sont concaténées sans doublon.
    """
    merged, divergences = {}, {}
    for part in partials:
        if not isinstance(part, dict):
            continue
        for key, value in part.items():
            if value in (None, "", [], {}):
                continue
            if key not in merged or merged[key] in (None, "", [], {}):
                merged[key] = value
            elif isinstance(merged[key], list) and isinstance(value, list):
                for item in value:
                    if item not in merged[key]:
                        merged[key].append(item)
            elif merged[key] != value:
                divergences.setdefault(key, [merged[key]])
                if value not in divergences[key]:
                    divergences[key].append(value)
    if divergences:
        merged["champs_divergents"] = divergences
    return merged


def _chunk(seq: list, size: int) -> list:
    """Découpe une liste en groupes d'au plus `size` éléments."""
    size = max(1, size)
    return [seq[i:i + size] for i in range(0, len(seq), size)]

In [ ]:
def _extract_structured_with_retries(pages: list, ocr_hint: str) -> tuple:
    """Interroge le modèle sur un groupe de pages, avec un escalier de nouvelles tentatives.

    Escalier : tentative normale -> tentatives aux résolutions de RETRY_RESOLUTIONS (Partie 1) -> échec.
    Réduire la résolution est la seule variable qu'il est utile de faire varier automatiquement ici ; les
    autres causes possibles d'une réponse dégénérée (chemin de calcul FP8, précision) ne se corrigent pas
    en cours d'exécution, elles demandent un rechargement du modèle (voir MODEL_LOAD_MODE, Partie 1).

    Retourne (dict_structuré, note_de_tentative). Lève une exception si toutes les tentatives échouent.
    """
    prompt = build_extraction_prompt(ocr_hint)
    attempts = [("résolution normale", None)] + [(f"résolution réduite {r}px", r) for r in RETRY_RESOLUTIONS]

    last_error = None
    for label, forced_resolution in attempts:
        try:
            batch = pages if forced_resolution is None else [
                resize_for_model(p, max_dimension=forced_resolution) for p in pages
            ]
            raw = _ask_model(batch, prompt, MAX_NEW_TOKENS_EXTRACTION)
            if _looks_degenerate(raw):
                last_error = f"réponse dégénérée ({raw[:20]!r}) à la tentative « {label} »"
                logger.warning("Tentative « %s » : réponse dégénérée, nouvel essai éventuel.", label)
                continue
            return extract_json_from_text(raw), label
        except Exception as e:                      # JSON illisible, erreur d'inférence...
            last_error = f"{type(e).__name__}: {e} (tentative « {label} »)"
            logger.warning("Tentative « %s » échouée : %s", label, e)
            continue

    raise ValueError(f"Toutes les tentatives ont échoué. Dernière erreur : {last_error}")

In [ ]:
def process_one_client(client_id: str) -> dict:
    """Traite le JUSTIFICATIF IDENTITE.PDF d'un client, avec une stratégie en cascade (voir Partie 6).

    Ne lève jamais d'exception : les échecs sont capturés et consignés dans le résultat retourné
    (statut="echec"), pour ne jamais interrompre le traitement par lot.
    """
    t0 = time.time()
    result = {"client_id": client_id, "statut": "echec", "erreur": None, "nb_pages_traitees": 0,
              "methode": None}
    pdf_path = FILTERED_DIR / client_id / IDENTITY_FILE

    try:
        if not pdf_path.exists():
            raise FileNotFoundError(f"Fichier introuvable : {pdf_path}")

        # --- Étape A : raccourci "couche texte native" (ni modèle, ni OCR, ni traitement d'image) ---
        text_layer = extract_pdf_text_layer(pdf_path) if PREFER_TEXT_LAYER else ""
        if text_layer:
            logger.info("[%s] couche texte native détectée (%d caractères) — lecture directe.",
                        client_id, len(text_layer))
            structured, note = _extract_structured_with_retries([], text_layer)
            result.update(structured)
            result["methode"] = "couche_texte_native"
            result["note_tentative"] = note
            result["statut"] = "succes"
            result["duree_secondes"] = round(time.time() - t0, 1)
            return result

        # --- Étape B : vrai scan -> rendu image + prétraitement complet (Partie 4) ---
        pages = pdf_to_images(pdf_path)
        if not pages:
            raise ValueError("Le PDF ne contient aucune page exploitable.")

        processed_pages = [preprocess_page(page_img) for page_img in pages]
        result["nb_pages_traitees"] = len(processed_pages)
        result["mode_couleur"] = [detect_color_mode(p) for p in processed_pages]

        ocr_texts = [ocr_raw_text(p) for p in processed_pages]
        ocr_combined = "\n\n".join(t for t in ocr_texts if t)
        result["ocr_diagnostic_longueur"] = len(ocr_combined)

        # --- Étape C : appel(s) au modèle, par groupes de pages, avec nouvelles tentatives ---
        groups = _chunk(processed_pages, MAX_PAGES_PER_CALL)
        try:
            partials, notes = [], []
            for i, group in enumerate(groups, start=1):
                hint = "\n\n".join(ocr_texts[(i - 1) * MAX_PAGES_PER_CALL: i * MAX_PAGES_PER_CALL])
                structured, note = _extract_structured_with_retries(group, hint)
                partials.append(structured)
                notes.append(f"groupe {i}/{len(groups)} : {note}")

            result.update(merge_extractions(partials))
            result["methode"] = "modele_vision" + (f" ({len(groups)} appels fusionnés)" if len(groups) > 1 else "")
            result["note_tentative"] = " | ".join(notes)
            result["statut"] = "succes"

        # --- Étape D : repli OCR seul, si le modèle a définitivement échoué ---
        except Exception as model_error:
            if len(ocr_combined) > 50:
                logger.warning("[%s] modèle indisponible/dégénéré — repli sur l'OCR brut.", client_id)
                result["methode"] = "repli_ocr_seul"
                result["texte_brut_ocr"] = ocr_combined
                result["statut"] = "succes_partiel"
                result["erreur"] = (
                    f"Extraction structurée impossible ({model_error}). Le texte OCR brut a été conservé "
                    "ci-dessus pour permettre une saisie/vérification manuelle. Une réponse dégénérée "
                    "ALORS QUE l'OCR fonctionne indique un problème MODÈLE/ENVIRONNEMENT (voir "
                    "MODEL_LOAD_MODE, Partie 1, et l'escalier de diagnostic, Partie 5bis-A) et non un "
                    "problème de qualité de ce document."
                )
            else:
                raise

    except Exception as e:
        result["erreur"] = str(e)
        logger.error("[%s] échec de l'extraction : %s", client_id, e)

    result["duree_secondes"] = round(time.time() - t0, 1)
    return result

## Partie 7 — Traitement par lot des clients cibles

Chaque client traité produit immédiatement son fichier `RESULTS_DIR/<client_id>.json`. Si le notebook est
interrompu puis relancé, les clients déjà traités sont automatiquement ignorés (sauf `FORCE_REPROCESS =
True` en Partie 1) : le traitement reprend là où il s'était arrêté.

**Sur la vitesse** : la durée de chaque client s'affiche en direct ci-dessous (log + barre de
progression) dès le premier traité — un moyen simple de repérer une lenteur anormale sans attendre la
fin du lot. Si le premier client est déjà beaucoup plus lent qu'attendu, revérifiez d'abord le
diagnostic de répartition GPU/CPU (Partie 5) avant de suspecter autre chose : c'est de loin le facteur
le plus déterminant. Pour aller plus loin (traiter plusieurs clients en un seul appel batché au modèle,
ou chevaucher le prétraitement CPU du client suivant avec l'inférence GPU du client en cours), voir la
note en fin de Partie 8 — non implémenté ici car cela ajoute une vraie complexité (gestion du padding
entre documents de tailles différentes, pression mémoire GPU) qu'il vaut mieux valider dans votre
environnement réel plutôt que de figer une hypothèse non testable ici.

In [ ]:
n_succes = n_echec = 0
progress = tqdm(target_clients, desc="Extraction JUSTIFICATIF IDENTITE")
for client_id in progress:
    out_path = RESULTS_DIR / f"{client_id}.json"
    if out_path.exists() and not FORCE_REPROCESS:
        continue  # déjà traité lors d'une exécution précédente

    result = process_one_client(client_id)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    logger.info("[%s] statut=%s (%ss)", client_id, result["statut"], result["duree_secondes"])

    if result["statut"] == "succes":
        n_succes += 1
    else:
        n_echec += 1
    progress.set_postfix(succes=n_succes, echec=n_echec)

print(f"\nTerminé. {n_succes} succès / {n_echec} échec(s) sur ce passage. "
      f"Résultats individuels disponibles dans {RESULTS_DIR.resolve()}")

## Partie 8 — Consolidation des résultats

On rassemble tous les fichiers JSON individuels en un seul export JSON et un export CSV (pratique pour
Excel / votre équipe conformité), puis on affiche un résumé et la liste des éventuels échecs à examiner
manuellement.

In [ ]:
all_results = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    with open(f, encoding="utf-8") as fh:
        all_results.append(json.load(fh))

if not all_results:
    print("Aucun résultat à consolider pour le moment (le traitement par lot n'a peut-être pas encore été exécuté).")
else:
    with open(COMBINED_RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    results_df = pd.json_normalize(all_results)
    results_df.to_csv(COMBINED_RESULTS_CSV, index=False, encoding="utf-8-sig")

    nb_succes = int((results_df["statut"] == "succes").sum())
    nb_echec = int((results_df["statut"] == "echec").sum())
    print(f"{nb_succes} succès / {nb_echec} échec(s) sur {len(results_df)} client(s) cible(s) traité(s).")

    if nb_echec:
        print("\nClients en échec (à vérifier manuellement) :")
        print(results_df.loc[results_df["statut"] == "echec", ["client_id", "erreur"]].to_string(index=False))

    logger.info("Consolidation terminée -> %s / %s", COMBINED_RESULTS_JSON, COMBINED_RESULTS_CSV)

results_df.head(10) if all_results else None

## Conclusion & prochaines étapes

- Le rapport d'inventaire (`rapport_inventaire_kyc.csv`) couvre les **5** types de documents ; seule
  l'extraction (Parties 4 à 8) se limite pour l'instant à `JUSTIFICATIF IDENTITE.PDF`, comme demandé.
- Pour traiter un autre type de document (`JUSTIFICATIF DOMICILE.PDF`, `FATCA.PDF`...), dupliquez les
  Parties 6 à 8 en adaptant `IDENTITY_FILE` et le schéma JSON du prompt aux champs propres à ce document.
- Avant un passage à l'échelle, validez la qualité de l'extraction sur un échantillon (10-20 clients) en
  comparant manuellement `resultats_extraction_identite.csv` aux documents sources, et affinez le prompt
  si nécessaire — en particulier pour les champs qui reviennent souvent dans `champs_incertains`.
- Chaque résultat contient maintenant aussi `mode_couleur` (liste par page, "couleur"/"noir_et_blanc") et
  `ocr_diagnostic_longueur` (caractères extraits par Tesseract) : utile en audit pour repérer si certains
  profils de documents (N&B, faible OCR) échouent plus souvent que d'autres.
- Vérifiez que ce traitement s'inscrit bien dans votre politique de confidentialité des données / conformité
  RGPD (ou équivalent local), notamment sur la durée de conservation des fichiers intermédiaires générés
  (`kyc_pipeline_workdir/`).
- **Pour aller plus loin sur la vitesse** (Partie 7), au-delà des optimisations déjà en place
  (redimensionnement avant le modèle, `attn_implementation="sdpa"`, diagnostic GPU/CPU) : un vrai gain
  supplémentaire viendrait soit du **traitement par lot batché** (plusieurs clients en un seul appel
  `generate`), soit du **chevauchement** du prétraitement CPU du client suivant avec l'inférence GPU du
  client en cours (pendant que le GPU travaille sur un client, préparer les images du suivant). Les deux
  ajoutent une complexité réelle (le batché demande de gérer un nombre de pages variable par client via
  du padding/attention mask ; le chevauchement demande une file d'attente/un thread dédié) qu'il est
  préférable de valider directement dans votre environnement Domino plutôt que de figer ici une
  hypothèse non testable sans accès à votre GPU réel.